# INF6083 - Projet P1
## Analyse du dataset Amazon Reviews 2023 - Books
### Équipe 7


## 0.1 Importation des bibliothèques

In [ ]:
# ── Bibliothèques standard ──────────────────────────────────────
import gc
import glob
import json
import os
import random
from tabnanny import check
import time
from pathlib import Path




# ── Calcul scientifique ──────────────────────────────────────────
import numpy as np
import pandas as pd
import polars as pl
from scipy.sparse import csr_matrix, save_npz, load_npz
import pynvml


# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ── PyTorch (GPU ou CPU) ─────────────────────────────────────────
import torch

import sys
print(sys.executable)

# ── RAPIDS (GPU) — optionnel, nécessaire uniquement pour les
#    cellules d'échantillonnage (Tâche 0). Si absent, ces cellules
#    seront sautées ; les tâches d'analyse (1-5) fonctionnent sans.
RAPIDS_AVAILABLE = False
try:
    import cudf
    import cupy as cp
    import rmm
    RAPIDS_AVAILABLE = True
except ImportError:
    cp = None

print(f"Is cudf, cupy and rmm imported succesfully? : {RAPIDS_AVAILABLE}")


def flush_ram():
    """Libère les objets Python non référencés."""
    gc.collect()

def flush_gpu():
    """Libère la mémoire GPU (cudf/cupy). À appeler après des opérations GPU lourdes."""
    gc.collect()
    if RAPIDS_AVAILABLE:
        try:
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
            cp.cuda.runtime.deviceSynchronize()
        except Exception:
            pass



# ── Global Variable ──────────────────────────────────────────────

# Paths & data layout
RAW_DATA_DIR = "data/raw/"
PROCESSED_DATA_DIR = "data/processed/"
RAW_JSONL_GLOB = f"{RAW_DATA_DIR}/jsonl/*.jsonl"
RAW_PARQUET_GLOB = f"{RAW_DATA_DIR}/parquet/*.parquet"
RAW_BOOKS_PATH = f"{RAW_DATA_DIR}/parquet/Books.parquet"
RAW_JSONL_PATHS = sorted(glob.glob(RAW_JSONL_GLOB))
RAW_PARQUET_PATHS = sorted(glob.glob(RAW_PARQUET_GLOB))


# Conversion JSONL → Parquet
# Colonnes pouvant contenir « — » ou d'autres valeurs non numériques dans le JSONL Amazon.
CANDIDATE_NUMERIC_COLUMNS = [
    "rating", "helpful_vote",           # Livres (avis)
    "average_rating", "rating_number", "price",  # meta_Books (métadonnées)
]
N_SPOT_CHECK = 100_000


# Sampling (Task 0)
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42
CHUNK_SIZE = 2_000_000
TARGET_YEARS = [2020, 2021, 2022, 2023]
TARGET_TOTAL = 2_000_000


# Sample file naming
SAMPLE_GLOB_ORIGINAL = f"{PROCESSED_DATA_DIR}sample-*/*_original.parquet"
SAMPLE_GLOB_CLEANED = f"{PROCESSED_DATA_DIR}sample-*/*_cleaned.parquet"
SAMPLE_GLOB_FILTERED = f"{PROCESSED_DATA_DIR}sample-*/*_filtered.parquet"
SPLIT_SUBDIR = "splits"
TRAIN_FILENAME = "train.parquet"
TEST_FILENAME = "test.parquet"


# Cleaning & filtering
MIN_DATE = pd.Timestamp("1995-07-01")
MAX_DATE = pd.Timestamp("2025-12-31")
MIN_RATINGS_USER = 20
MIN_RATINGS_BOOK = 5
MAX_ITER = 20


# Train/test split
TRAIN_RATIO = 0.80
TEST_RATIO = 1.0 - TRAIN_RATIO

## 0.2 Chargement des données (JSONL → Parquet)
Transformation du dataset en base Parquet pour un chargement rapide (10–50×). Fichier source : `data/Books.jsonl`.

In [ ]:
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║  CONVERSION JSONL → PARQUET                                         ║")
print("║  Lecture des fichiers JSONL bruts et conversion en Parquet (Polars). ║")
print("║  Le format Parquet offre une compression columnar 3-5× plus         ║")
print("║  compacte que JSONL et permet la lecture paresseuse (lazy scanning). ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print(f"  Fichiers JSONL trouvés : {len(RAW_JSONL_PATHS)}")
print(f"  Colonnes numériques surveillées : {CANDIDATE_NUMERIC_COLUMNS}")

for jsonl_path in RAW_JSONL_PATHS:
    parquet_path = jsonl_path.replace("jsonl", "parquet")
    
    # ── 1. Passer si déjà fait (avec vérification d'intégrité) ───────────
    if os.path.exists(parquet_path):
        try:
            n_parquet = pl.scan_parquet(parquet_path).select(pl.len()).collect().item()
            cols_parquet = set(pl.scan_parquet(parquet_path).collect_schema().names())
            
            with open(jsonl_path, "rb") as f:
                n_jsonl = sum(1 for line in f if line.strip())
            
            if n_parquet != n_jsonl:
                print(f"  ⚠ Nombre de lignes incohérent, reconversion : {jsonl_path}")
            else:
                schema_jsonl = pl.scan_ndjson(jsonl_path, infer_schema_length=1).collect_schema()
                cols_jsonl = set(schema_jsonl.names())
                if cols_jsonl == cols_parquet:
                    print(f"  ✓ Déjà converti (vérifié) : {parquet_path}")
                    continue
                else:
                    print(f"  ⚠ Schéma incohérent, reconversion : {jsonl_path}")
        except Exception as e:
            print(f"  ⚠ Fichier existant invalide ({e}), reconversion : {parquet_path}")
    
    # ── 2. Détection des colonnes à surcharger ───────────────────────────
    schema_initial = pl.scan_ndjson(jsonl_path, infer_schema_length=1).collect_schema()
    cols_to_override = [c for c in schema_initial.names() if c in CANDIDATE_NUMERIC_COLUMNS]
    schema_overrides = {c: pl.Utf8 for c in cols_to_override} if cols_to_override else None
    
    # ── 3. Conversion ────────────────────────────────────────────────────
    print(f"  Conversion {jsonl_path} → {parquet_path}...")
    kwargs = {"schema_overrides": schema_overrides} if schema_overrides else {}
    lf = pl.scan_ndjson(jsonl_path, **kwargs)
    if cols_to_override:
        lf = lf.with_columns([
            pl.col(c).cast(pl.Float64, strict=False) for c in cols_to_override
        ])
    lf.sink_parquet(parquet_path)
    del lf
    gc.collect()
    
    # ── 4. Intégrité : nombre de lignes (sans chargement complet) ────────
    with open(jsonl_path, "rb") as f:
        n_jsonl = sum(1 for line in f if line.strip())
    n_parquet = pl.scan_parquet(parquet_path).select(pl.len()).collect().item()
    
    if n_jsonl != n_parquet:
        raise ValueError(
            f"Incohérence de données ! JSONL : {n_jsonl:,} lignes, Parquet : {n_parquet:,} lignes. "
            f"Fichier : {jsonl_path}"
        )
    print(f"  ✓ Lignes vérifiées : {n_parquet:,}")
    
    # ── 5. Intégrité : schéma (noms de colonnes) ─────────────────────────
    cols_jsonl = set(pl.scan_ndjson(jsonl_path, infer_schema_length=1).collect_schema().names())
    cols_parquet = set(pl.scan_parquet(parquet_path).collect_schema().names())
    
    if cols_jsonl != cols_parquet:
        only_jsonl = cols_jsonl - cols_parquet
        only_parquet = cols_parquet - cols_jsonl
        raise ValueError(
            f"Schéma incohérent ! {jsonl_path}\n"
            f"  Uniquement dans JSONL : {only_jsonl or 'aucun'}\n"
            f"  Uniquement dans Parquet : {only_parquet or 'aucun'}"
        )
    print(f"  ✓ Schéma vérifié : {list(cols_parquet)}")
    
    # ── 6. Vérification par échantillon ──────────────────────────────────
    n_sample = min(N_SPOT_CHECK, n_parquet)
    if n_sample == 0:
        print(f"  ✓ Vérification ignorée (fichier vide)")
    else:
        read_kwargs = {"schema_overrides": schema_overrides} if schema_overrides else {}
        df_jsonl_sample = pl.read_ndjson(jsonl_path, n_rows=n_sample, **read_kwargs)
        if cols_to_override:
            df_jsonl_sample = df_jsonl_sample.with_columns([
                pl.col(c).cast(pl.Float64, strict=False) for c in cols_to_override
            ])
        df_parquet_sample = pl.read_parquet(parquet_path, n_rows=n_sample)
        df_jsonl_sample = df_jsonl_sample.select(df_parquet_sample.columns)
        
        for col in df_parquet_sample.columns:
            s_jsonl = df_jsonl_sample[col]
            s_parquet = df_parquet_sample[col]
            if not s_jsonl.eq_missing(s_parquet).all():
                diff_mask = ~s_jsonl.eq_missing(s_parquet)
                idx = diff_mask.arg_true()[0]
                raise ValueError(
                    f"Vérification échouée : colonne '{col}' diffère à la ligne {idx}\n"
                    f"  JSONL :   {s_jsonl[idx]}\n"
                    f"  Parquet : {s_parquet[idx]}"
                )
        print(f"  ✓ Vérification réussie ({n_sample:,} premières lignes)")
        
        del df_jsonl_sample, df_parquet_sample
        gc.collect()
    
    # ── 7. Nettoyage par fichier ─────────────────────────────────────────
    gc.collect()

# ── 8. Nettoyage final et résumé ────────────────────────────────────
gc.collect()
print("\n✓ Conversion terminée. Tous les fichiers JSONL ont été convertis en Parquet.")
print("  Les fichiers Parquet sont prêts pour l'analyse dans les cellules suivantes.")

# 3.1 Tâche 0 - Chargement et échantillonnage des données

## 3.1.1 Échantillonnage stratégique

### 1) Echantillonnage des utilisateurs actifs (≥ 20 reviews)
- Filtrage des utilisateurs actifs (≥ 20 reviews)
- Sélection aléatoire de 50,000 utilisateurs
- Conservation de toutes leurs interactions

In [ ]:
# Guard : cette cellule nécessite RAPIDS (cudf/cupy/rmm)
if not RAPIDS_AVAILABLE:
    print("ℹ RAPIDS (cudf/cupy/rmm) non installé — cellule ignorée.")
else:
    flush_ram()
    flush_gpu()

    # ── Aide : libérer toute la mémoire ──────────────────────────────
    def flush_memory():
        """Libération agressive de la RAM et de la VRAM."""
        # 1. Ramasse-miettes Python — libère les objets non référencés
        gc.collect()

        # 2. Pool mémoire CuPy — libère les blocs GPU en cache
        mempool = cp.get_default_memory_pool()
        pinned_mempool = cp.get_default_pinned_memory_pool()
        mempool.free_all_blocks()
        pinned_mempool.free_all_blocks()

        # 3. Synchronisation CUDA forcée (s'assurer que toutes les opérations GPU sont terminées)
        cp.cuda.runtime.deviceSynchronize()

    def print_memory_status(label=""):
        """Affiche l'utilisation courante de la mémoire GPU."""
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        print(f"  [{label}] GPU: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB "
              f"(free: {info.free/1e9:.2f} GB)")
        pynvml.nvmlShutdown()


    # ── Configuration du pool mémoire RMM ────────────────────────────
    rmm.reinitialize(
        managed_memory=True,
        pool_allocator=True,
    )

    print_memory_status("Before start")

    # ================================================================
    # PHASE 1 : Charger le JSONL, compter par utilisateur, trouver les actifs
    # ================================================================
    start = time.time()
    print("Phase 1 : Chargement en mémoire GPU...")

    gdf = cudf.read_parquet(RAW_BOOKS_PATH)
    gdf['rating'] = gdf['rating'].astype('int8')

    print_memory_status("After load")
    print(f"  GPU DataFrame: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    # Comptage des reviews par utilisateur sur GPU
    user_counts = gdf['user_id'].value_counts()
    active_users = user_counts[user_counts >= MIN_REVIEWS].index

    # ── Transférer les IDs des utilisateurs actifs vers le CPU ───────
    active_list = active_users.to_pandas().tolist()
    print(f"  Active users: {len(active_list):,}")

    # ── Échantillonner 50 000 utilisateurs aléatoirement (CPU) ──────
    random.seed(SEED)
    selected_users = random.sample(active_list, min(NUM_USERS, len(active_list)))

    # ── FLUSH : supprimer les comptages, on ne garde que la liste d'IDs ─
    del user_counts, active_users, active_list
    flush_memory()
    print_memory_status("After count flush")

    # ================================================================
    # PHASE 2 : Filtrer le DataFrame complet pour les utilisateurs échantillonnés
    # ================================================================
    print("\nPhase 2 : Filtrage...")

    # Reconvertir en cudf Series pour isin() côté GPU
    selected_series = cudf.Series(selected_users)
    mask = gdf['user_id'].isin(selected_series)
    sample_gdf = gdf[mask]

    print(f"  Reviews correspondantes : {len(sample_gdf):,}")

    # ── FLUSH : supprimer le DataFrame complet — on n'en a plus besoin ──
    del gdf, mask, selected_series
    flush_memory()
    print_memory_status("After filter flush")

    # ================================================================
    # PHASE 3 : Transfert vers CPU et sauvegarde
    # ================================================================
    print("\nPhase 3 : Sauvegarde dans un .parquet")


    elapsed = time.time() - start
    print(f"\nTemps d'execution: {elapsed:.2f}s")
    print(f"  Reviews échantillonnées : {len(sample_gdf):,}")
    print(f"  Utilisateurs uniques : {sample_gdf['user_id'].nunique():,}")

    # Save
    sample_gdf.to_parquet(f'{PROCESSED_DATA_DIR}/sample_gpu_active_users_original.parquet', compression='snappy')


    # ── FLUSH : supprimer le DataFrame GPU — les données sont sur CPU ──
    del sample_gdf
    flush_memory()
    print_memory_status("After GPU->Parquet flush")

    gc.collect()
    flush_gpu()
    flush_memory()
    flush_ram()
    print_memory_status("Final cleanup")



### 2) Echantillonage temporel
- Filtrage des reviews ayant eu lieu entre 2020-01-01 et 2023-12-31
- Sélection aléatoire de user ayant au moins 20 reviews
- Conservation de toutes les interactions


In [ ]:
# Guard : cette cellule nécessite RAPIDS (cudf/cupy/rmm)
if not RAPIDS_AVAILABLE:
    print("ℹ RAPIDS (cudf/cupy/rmm) non installé — cellule ignorée.")
else:
    flush_ram()
    flush_gpu()

    # Configuration de la mémoire GPU
    rmm.reinitialize(
        managed_memory=True,  # Unified memory CPU/GPU
        pool_allocator=True
    )

    def sample_with_gpu(path = str):
        """
        Échantillonnage GPU-accéléré avec cuDF (RAPIDS)

        Performance: 10-50x plus rapide que pandas
        Requis: GPU NVIDIA avec 8GB+ VRAM

        """
        flush_memory()
        print("Chargement en mémoire GPU...")
        monitor_gpu_memory()
        gdf = cudf.read_parquet(path)
        monitor_gpu_memory()

        gdf['rating'] = gdf['rating'].astype('int8')

        print(f"  Mémoire GPU utilisée : {gdf.memory_usage(deep=True).sum() / 1e9:.2f} Go")

        gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'], unit='ms')
        gdf['year'] = gdf['timestamp'].dt.year

        # ── Diagnostics année par année ──────────────────────────────────
        gdf_target = gdf[gdf['year'].isin(TARGET_YEARS)]
        year_stats = (
            gdf_target.groupby('year')
            .agg({
                'user_id': ['count', 'nunique'],
                'rating': 'mean'
            })
        )
        # Transfert vers pandas pour affichage
        ys = year_stats.to_pandas()
        ys.columns = ['review_count', 'unique_users', 'avg_rating']
        ys = ys.sort_index()
        print(f"\n── Répartition par année ({TARGET_YEARS[0]}–{TARGET_YEARS[-1]}) ──")
        print(ys.to_string())
        print(f"\nTotal reviews : {ys['review_count'].sum():,}")
        print(f"  Utilisateurs uniques (total, avec chevauchement) : {ys['unique_users'].sum():,}")

        # ① Restreindre à la période cible
        gdf_period = gdf[gdf['year'].isin(TARGET_YEARS)]
        del gdf
        monitor_gpu_memory()

        print(f"  Reviews dans {TARGET_YEARS[0]}–{TARGET_YEARS[-1]} : {len(gdf_period):,}")

        # ② Compter les reviews par utilisateur dans la période
        user_counts = gdf_period['user_id'].value_counts().reset_index()
        user_counts.columns = ['user_id', 'review_count']

        # ③ Ne garder que les utilisateurs avec >= MIN_REVIEWS dans cette période
        active_in_period = user_counts[user_counts['review_count'] >= MIN_REVIEWS]
        print(f"Active users (>= {MIN_REVIEWS} reviews in period): {len(active_in_period):,}")

        # ④ Échantillonner jusqu'à 50 000 utilisateurs
        active_list = active_in_period['user_id'].to_pandas().tolist()
        random.seed(SEED)
        n_to_sample = min(NUM_USERS, len(active_list))
        sampled_users = random.sample(active_list, n_to_sample)
        print(f"  Utilisateurs échantillonnés : {n_to_sample:,} / {len(active_list):,}")

        # ⑤ Collecter TOUTES leurs reviews dans la période (pas de plafond)
        sampled_series = cudf.Series(sampled_users)
        sample_gdf = gdf_period[gdf_period['user_id'].isin(sampled_series)]
        monitor_gpu_memory()

        print(f"  Reviews : {len(sample_gdf):,} provenant de {sample_gdf['user_id'].nunique():,} utilisateurs")
        print(f"  Moy. reviews/utilisateur : {len(sample_gdf) / sample_gdf['user_id'].nunique():.1f}")
        del gdf_period


        return sample_gdf

    # Suivi de la mémoire GPU
    def monitor_gpu_memory():
        """Affiche utilisation GPU"""
        import pynvml

        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)

        print(f"  Mémoire GPU : {info.used/1e9:.2f}/{info.total/1e9:.2f} Go")
        pynvml.nvmlShutdown()

    # ── Aide : libérer toute la mémoire ──────────────────────────────
    def flush_memory():
        """Aggressively free both RAM and VRAM."""
        # 1. Python garbage collector — release unreferenced objects
        gc.collect()

        # 2. Pool mémoire CuPy — libère les blocs GPU en cache
        mempool = cp.get_default_memory_pool()
        pinned_mempool = cp.get_default_pinned_memory_pool()
        mempool.free_all_blocks()
        pinned_mempool.free_all_blocks()
        # 3. Synchronisation CUDA forcée (s'assurer que toutes les opérations GPU sont terminées)
        cp.cuda.runtime.deviceSynchronize()

    # Utilisation
    if __name__ == '__main__':
        import time
        start = time.time()
        sample = sample_with_gpu(RAW_BOOKS_PATH)
        elapsed = time.time() - start

        print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
        print(f"   Reviews échantillonnées: {len(sample):,}")

        # Sauvegarde
        sample.to_parquet(f'{PROCESSED_DATA_DIR}/sample_gpu_temporal_original.parquet', compression='snappy')
        del sample
        flush_memory()
        monitor_gpu_memory()  # after transferring to CPU and freeing GPU
        gc.collect()


## 3.1.3 Prétraitement


- Nettoyage des ratings
- Filtrage utilisateurs/items
- Construction matrice utilisateur-item (CSR)
- Split train/test (80/20 stratifié)

### 1) Nettoyage des Données

 - Supprimer les reviews sans note
 - Traiter les timestamps invalides
 - Convertir le rating en float

In [ ]:
flush_ram()
flush_gpu()

# ══════════════════════════════════════════════════════════════════════
# NETTOYAGE DES DONNÉES (Data Preprocessing)
#
# Ce script parcourt tous les fichiers Parquet d'échantillons et
# applique trois étapes de nettoyage :
#   1. Suppression des reviews sans note (rating manquant ou invalide)
#   2. Filtrage des timestamps invalides (hors plage temporelle)
#   3. Conversion du rating en float pour cohérence numérique
#
# À la fin, chaque fichier est sauvegardé en version nettoyée.
# ══════════════════════════════════════════════════════════════════════


# Amazon a été fondé en juillet 1995 — aucune review ne peut être antérieure.
# On borne aussi par le futur pour exclure les timestamps manifestement erronés.
MIN_DATE = pd.Timestamp("1995-07-01")
MAX_DATE = pd.Timestamp("2025-12-31")

for path in SAMPLE_GLOB_ORIGINAL:
    # ── Garde : fichiers trop petits (probablement corrompus) ──────
    if os.path.getsize(path) < 1024:
        print(f"  ⚠ Fichier ignoré : {path} ({os.path.getsize(path)} octets — probablement corrompu)")
        continue

    print(f"\n{'═' * 60}")
    print(f"  Fichier : {path}")
    print(f"{'═' * 60}")

    df = pd.read_parquet(path)
    n_original = len(df)

    if n_original == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    print(f"  Nombre initial de reviews : {n_original:,}")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Suppression des reviews sans note valide
    # ──────────────────────────────────────────────────────────────────
    # Deux cas possibles dans le jeu de données Amazon :
    #   - Le champ 'rating' est absent du JSON original → NaN en Pandas
    #   - Le champ 'rating' vaut 0, ce qui est hors de l'échelle 1–5
    # Dans les deux cas, la review est inutilisable pour un système de
    # recommandation, on la supprime.
    # ──────────────────────────────────────────────────────────────────

    n_nan_rating = df["rating"].isna().sum()
    n_zero_rating = (df["rating"] == 0).sum()

    df = df.dropna(subset=["rating"])
    df = df[df["rating"] > 0]
    n_after_rating = len(df)
    n_dropped_rating = n_original - n_after_rating

    print(f"\n  ── Étape 1 : Nettoyage des notes (rating) ──")
    print(f"     Reviews avec rating NaN   : {n_nan_rating:,}")
    print(f"     Reviews avec rating == 0  : {n_zero_rating:,}")
    print(f"     Total supprimées          : {n_dropped_rating:,}")
    print(f"     Reviews restantes         : {n_after_rating:,}")
    if n_dropped_rating == 0:
        print(f"     ✓ Aucune review sans note — le jeu de données est propre sur ce critère.")
    else:
        print(f"     → {n_dropped_rating / n_original * 100:.2f}% des reviews n'avaient pas de note valide.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Filtrage des timestamps invalides
    # ──────────────────────────────────────────────────────────────────
    # Les timestamps sont stockés en millisecondes Unix (ms depuis
    # le 1er janvier 1970). On les convertit en datetime pour vérifier
    # qu'ils tombent dans une plage plausible :
    #   - Borne inférieure : 1er juillet 1995 (fondation d'Amazon)
    #   - Borne supérieure : 31 décembre 2025
    #
    # errors="coerce" transforme les valeurs non convertibles en NaT
    # (Not a Time), qui sont ensuite exclues par le filtre between().
    #
    # Cas détectés :
    #   - Timestamps nuls ou manquants → NaT après conversion
    #   - Timestamps négatifs → dates avant 1970, hors plage
    #   - Timestamps en secondes au lieu de millisecondes → dates
    #     autour de 1970, également hors plage
    #   - Timestamps dans le futur → données erronées
    # ──────────────────────────────────────────────────────────────────

    n_null_ts = df["timestamp"].isna().sum()
    df["_date"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
    n_nat = df["_date"].isna().sum()
    n_before_min = (df["_date"] < MIN_DATE).sum()
    n_after_max = (df["_date"] > MAX_DATE).sum()

    df = df[df["_date"].between(MIN_DATE, MAX_DATE)]
    df.drop(columns=["_date"], inplace=True)
    n_after_ts = len(df)
    n_dropped_ts = n_after_rating - n_after_ts

    print(f"\n  ── Étape 2 : Nettoyage des timestamps ──")
    print(f"     Timestamps nuls/manquants          : {n_null_ts:,}")
    print(f"     Non convertibles (→ NaT)           : {n_nat:,}")
    print(f"     Avant {MIN_DATE.date()} (pré-Amazon) : {n_before_min:,}")
    print(f"     Après {MAX_DATE.date()} (futur)      : {n_after_max:,}")
    print(f"     Total supprimées                   : {n_dropped_ts:,}")
    print(f"     Reviews restantes                  : {n_after_ts:,}")
    if n_dropped_ts == 0:
        print(f"     ✓ Tous les timestamps sont valides — aucune suppression nécessaire.")
    else:
        print(f"     → {n_dropped_ts / n_after_rating * 100:.2f}% des reviews avaient un timestamp invalide.")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Conversion du rating en float
    # ──────────────────────────────────────────────────────────────────
    # Selon la méthode d'écriture du Parquet, le rating peut être stocké
    # en int64, int32, ou déjà en float. On force float64 pour :
    #   - Garantir la cohérence entre tous les échantillons
    #   - Éviter les erreurs de division entière dans les calculs de
    #     moyennes et corrélations en aval
    # ──────────────────────────────────────────────────────────────────

    dtype_before = df["rating"].dtype
    df["rating"] = df["rating"].astype(float)
    dtype_after = df["rating"].dtype

    print(f"\n  ── Étape 3 : Conversion du type de rating ──")
    print(f"     Type avant conversion : {dtype_before}")
    print(f"     Type après conversion : {dtype_after}")
    if dtype_before == dtype_after:
        print(f"     ✓ Le rating était déjà en {dtype_after} — aucune conversion nécessaire.")
    else:
        print(f"     → Converti de {dtype_before} vers {dtype_after}.")

    # ══════════════════════════════════════════════════════════════════
    # RÉSUMÉ FINAL
    # ══════════════════════════════════════════════════════════════════

    n_final = len(df)
    n_total_dropped = n_original - n_final

    print(f"\n  {'─' * 56}")
    print(f"  RÉSUMÉ — {path}")
    print(f"  {'─' * 56}")
    print(f"     Reviews initiales     : {n_original:,}")
    print(f"     Supprimées (rating)   : {n_dropped_rating:,}")
    print(f"     Supprimées (timestamp): {n_dropped_ts:,}")
    print(f"     Reviews finales       : {n_final:,}")
    print(f"     Taux de rétention     : {n_final / n_original * 100:.2f}%")
    print(f"     Utilisateurs restants : {df['user_id'].nunique():,}")
    print(f"     Livres restants       : {df['parent_asin'].nunique():,}")
    print(f"  {'─' * 56}")

    # ── Sauvegarde (décommenter quand prêt) ────────────────────────
    out_path = path.replace("_original.parquet", "_cleaned.parquet")
    df.to_parquet(out_path, index=False)
    print(f"  ✓ Fichier nettoyé sauvegardé : {path}")
    del df

flush_ram()
flush_gpu()



### 2) Filtrage

- Nombre minimum de notes par utilisateur : 10–20
- Nombre minimum de notes par livre : 5–10
- Recalculer le taux de sparsité après filtrage

In [ ]:
flush_ram()
flush_gpu()

# ══════════════════════════════════════════════════════════════════════════
# FILTRAGE PAR SEUILS D'ACTIVITÉ (Threshold Filtering)
#
# Objectif : éliminer les utilisateurs et les livres ayant trop peu
# d'interactions. En filtrage collaboratif, un utilisateur avec 1–2 notes
# ne fournit aucun signal exploitable (problème du « cold start »), et un
# livre noté par un seul lecteur ne peut pas être recommandé par similarité.
#
# ── Choix des seuils ──────────────────────────────────────────────────────
#
# MIN_RATINGS_USER = 20
#   Notre échantillon « cudf-claude » a été construit en ne gardant que
#   les utilisateurs ayant ≥ 20 reviews dans le jeu COMPLET (~27M).
#   Toutefois, après le nettoyage précédent (rating/timestamps) et après
#   le filtrage des livres ci-dessous, certains utilisateurs pourraient
#   passer sous ce seuil. On réapplique donc 20 pour maintenir la
#   cohérence avec le critère d'échantillonnage initial.
#   ► Justification : 20 est le seuil standard dans la littérature
#     sur les systèmes de recommandation (cf. Koren 2008, He et al. 2017).
#     Il garantit un profil utilisateur assez riche pour que les algorithmes
#     de voisinage (kNN) ou de factorisation (SVD/ALS) convergent.
#
# MIN_RATINGS_BOOK = 5
#   Un livre noté par < 5 lecteurs est statistiquement « invisible » :
#   sa moyenne est instable et la co-occurrence avec d'autres livres est
#   trop faible pour contribuer aux vecteurs de similarité.
#   ► Justification : le seuil de 5 est couramment utilisé dans les
#     benchmarks Amazon (McAuley et al. 2015, Ni et al. 2019).
#     On privilégie 5 plutôt que 10 pour conserver un catalogue plus
#     large et limiter la perte de diversité (long tail).
#
# ── Filtrage itératif ─────────────────────────────────────────────────────
#
# IMPORTANT : le filtrage n'est PAS une opération unique.
# Supprimer des livres rares peut faire descendre certains utilisateurs
# sous le seuil, et inversement. On boucle donc jusqu'à convergence
# (plus aucune suppression à effectuer). En pratique, 2–4 itérations
# suffisent.
# ══════════════════════════════════════════════════════════════════════════



for path in SAMPLE_GLOB_CLEANED:
    if os.path.getsize(path) < 1024:
        print(f"  ⚠ Fichier ignoré : {path} (trop petit)")
        continue

    print(f"\n{'═' * 70}")
    print(f"  Fichier : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)
    n_before_filter = len(df)
    u_before = df["user_id"].nunique()
    b_before = df["parent_asin"].nunique()

    if n_before_filter == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # ──────────────────────────────────────────────────────────────────
    # Calcul de la sparsité AVANT filtrage
    # ──────────────────────────────────────────────────────────────────
    # La sparsité mesure le pourcentage de cases vides dans la matrice
    # utilisateur × livre. Formule :
    #   sparsité = 1 − |R| / (|U| × |I|)
    # où |R| = nombre de ratings, |U| = nb d'utilisateurs, |I| = nb de livres.
    #
    # Une sparsité de 99.99% signifie que seule 0.01% de la matrice est
    # remplie — c'est typique des grands jeux de données de recommandation.
    # ──────────────────────────────────────────────────────────────────

    sparsity_before = 1 - n_before_filter / (u_before * b_before)

    print(f"\n  ── État AVANT filtrage ──")
    print(f"     Reviews       : {n_before_filter:,}")
    print(f"     Utilisateurs  : {u_before:,}")
    print(f"     Livres        : {b_before:,}")
    print(f"     Sparsité      : {sparsity_before * 100:.4f}%")
    print(f"     Densité       : {(1 - sparsity_before) * 100:.6f}%")

    # ──────────────────────────────────────────────────────────────────
    # Diagnostic pré-filtrage : distribution de l'activité
    # ──────────────────────────────────────────────────────────────────
    # On affiche combien d'utilisateurs/livres seraient éliminés pour
    # donner une idée de l'impact avant de filtrer.
    # ──────────────────────────────────────────────────────────────────

    ratings_per_user = df.groupby("user_id").size()
    ratings_per_book = df.groupby("parent_asin").size()

    n_users_below = (ratings_per_user < MIN_RATINGS_USER).sum()
    n_books_below = (ratings_per_book < MIN_RATINGS_BOOK).sum()

    print(f"\n  ── Diagnostic pré-filtrage ──")
    print(f"     Seuil utilisateur : ≥ {MIN_RATINGS_USER} ratings")
    print(f"     Seuil livre       : ≥ {MIN_RATINGS_BOOK} ratings")
    print(f"     Utilisateurs sous le seuil : {n_users_below:,} / {u_before:,}"
          f" ({n_users_below / u_before * 100:.1f}%)")
    print(f"     Livres sous le seuil       : {n_books_below:,} / {b_before:,}"
          f" ({n_books_below / b_before * 100:.1f}%)")

    # ──────────────────────────────────────────────────────────────────
    # Filtrage itératif jusqu'à convergence
    # ──────────────────────────────────────────────────────────────────
    # À chaque itération :
    #   1. On supprime les livres ayant < MIN_RATINGS_BOOK notes
    #   2. On supprime les utilisateurs ayant < MIN_RATINGS_USER notes
    #   3. Si rien n'a changé → on a convergé, on arrête
    #
    # On limite à 20 itérations par sécurité (jamais atteint en pratique).
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  ── Filtrage itératif ──")
    MAX_ITER = 20

    for iteration in range(1, MAX_ITER + 1):
        n_start = len(df)

        # Filtrer les livres avec trop peu de notes
        book_counts = df.groupby("parent_asin").size()
        books_ok = book_counts[book_counts >= MIN_RATINGS_BOOK].index
        df = df[df["parent_asin"].isin(books_ok)]
        n_after_books = len(df)
        dropped_books_reviews = n_start - n_after_books

        # Filtrer les utilisateurs avec trop peu de notes
        user_counts = df.groupby("user_id").size()
        users_ok = user_counts[user_counts >= MIN_RATINGS_USER].index
        df = df[df["user_id"].isin(users_ok)]
        n_after_users = len(df)
        dropped_users_reviews = n_after_books - n_after_users

        total_dropped = n_start - n_after_users

        print(f"     Itération {iteration:>2d} : "
              f"−{dropped_books_reviews:,} (livres) "
              f"−{dropped_users_reviews:,} (users) "
              f"→ {n_after_users:,} reviews restantes")

        if total_dropped == 0:
            print(f"     ✓ Convergence atteinte à l'itération {iteration}.")
            break
    else:
        print(f"     ⚠ Limite de {MAX_ITER} itérations atteinte sans convergence.")

    # ──────────────────────────────────────────────────────────────────
    # Calcul de la sparsité APRÈS filtrage
    # ──────────────────────────────────────────────────────────────────

    n_after_filter = len(df)
    u_after = df["user_id"].nunique()
    b_after = df["parent_asin"].nunique()

    if u_after > 0 and b_after > 0:
        sparsity_after = 1 - n_after_filter / (u_after * b_after)
    else:
        sparsity_after = 1.0

    # ──────────────────────────────────────────────────────────────────
    # Résumé comparatif AVANT / APRÈS
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DU FILTRAGE — {path}")
    print(f"  {'─' * 66}")
    print(f"  {'':30s} {'AVANT':>14s}   {'APRÈS':>14s}   {'Δ':>10s}")
    print(f"  {'─' * 66}")

    delta_r = n_after_filter - n_before_filter
    delta_u = u_after - u_before
    delta_b = b_after - b_before

    print(f"  {'Reviews':30s} {n_before_filter:>14,}   {n_after_filter:>14,}   {delta_r:>+10,}")
    print(f"  {'Utilisateurs':30s} {u_before:>14,}   {u_after:>14,}   {delta_u:>+10,}")
    print(f"  {'Livres':30s} {b_before:>14,}   {b_after:>14,}   {delta_b:>+10,}")
    print(f"  {'Sparsité (%)':30s} {sparsity_before * 100:>13.4f}%   {sparsity_after * 100:>13.4f}%")
    print(f"  {'Densité (%)':30s} {(1 - sparsity_before) * 100:>13.6f}%   {(1 - sparsity_after) * 100:>13.6f}%")
    print(f"  {'─' * 66}")

    # ── Interprétation automatique des résultats ──────────────────────

    pct_reviews_kept = n_after_filter / n_before_filter * 100 if n_before_filter > 0 else 0
    pct_users_kept = u_after / u_before * 100 if u_before > 0 else 0
    pct_books_kept = b_after / b_before * 100 if b_before > 0 else 0
    density_gain = ((1 - sparsity_after) / (1 - sparsity_before) - 1) * 100 if sparsity_before < 1 else 0

    print(f"\n  ── Interprétation ──")
    print(f"     Taux de rétention des reviews       : {pct_reviews_kept:.1f}%")
    print(f"     Taux de rétention des utilisateurs   : {pct_users_kept:.1f}%")
    print(f"     Taux de rétention des livres         : {pct_books_kept:.1f}%")
    print(f"     Gain de densité                      : ×{density_gain / 100 + 1:.1f} ({density_gain:+.1f}%)")
    print()
    print(f"     La matrice U×I est passée de {u_before:,}×{b_before:,} = {u_before * b_before:,} cases")
    print(f"     à {u_after:,}×{b_after:,} = {u_after * b_after:,} cases.")
    print(f"     En éliminant les livres rares (< {MIN_RATINGS_BOOK} notes) et les utilisateurs")
    print(f"     peu actifs (< {MIN_RATINGS_USER} notes), on concentre le signal utile")
    print(f"     sur un sous-ensemble plus dense, ce qui améliore directement")
    print(f"     la qualité des recommandations par filtrage collaboratif.")

    # ── Sauvegarde (décommenter quand prêt) ────────────────────────
    out_path = path.replace("_cleaned.parquet", "_filtered.parquet")
    df.to_parquet(out_path, index=False)
    print(f"\n  ✓ Fichier filtré sauvegardé : {out_path }")
    del df

flush_ram()
flush_gpu()



### 4) Division en ensembles d'entraînement et de test

#### A) Division des données

Séparer les données en :
- **Ensemble d'entraînement** : 80 % des notes
- **Ensemble de test** : 20 % des notes
- **Stratification par utilisateur** : chaque utilisateur doit avoir au moins une note dans chaque ensemble

In [ ]:

# ══════════════════════════════════════════════════════════════════════════
# SÉPARATION TRAIN / TEST STRATIFIÉE PAR UTILISATEUR
#
# Objectif : diviser les ratings en 80% entraînement / 20% test,
# en garantissant que CHAQUE utilisateur possède au moins 1 rating
# dans chaque ensemble.
#
# ── Pourquoi stratifier par utilisateur ? ─────────────────────────────────
#
# En recommandation, on évalue la capacité du modèle à prédire les
# goûts d'un utilisateur CONNU à partir de son historique partiel.
# Si un utilisateur n'a aucun rating dans le train, le modèle ne peut
# rien apprendre de lui (cold start total). Si un utilisateur n'a
# aucun rating dans le test, on ne peut pas mesurer la qualité des
# prédictions le concernant.
#
# Un split aléatoire global (sans stratification) risquerait de
# concentrer tous les ratings d'un utilisateur rare dans un seul
# ensemble — d'où la nécessité de stratifier.
#
# ── Contrainte pour les utilisateurs avec peu de ratings ──────────────────
#
# Pour un utilisateur avec n ratings et un ratio test de 20% :
#   n_test  = max(1, floor(n × 0.2))   — au moins 1 dans le test
#   n_test  = min(n_test, n − 1)        — au moins 1 dans le train
#
# Cas limites :
#   n = 2 → n_test = 1, n_train = 1  (split 50/50, inévitable)
#   n = 5 → n_test = 1, n_train = 4  (split 80/20)
#   n = 10 → n_test = 2, n_train = 8  (split exact 80/20)
#   n = 50 → n_test = 10, n_train = 40  (split exact 80/20)
#
# Les utilisateurs avec n = 2 auront un split plus généreux pour le
# test (50/50 au lieu de 80/20), ce qui est le prix à payer pour
# respecter la contrainte « au moins 1 dans chaque ensemble ».
#
# ── Approche vectorisée (sans boucle Python sur les utilisateurs) ─────────
#
# L'approche naïve serait de boucler sur chaque utilisateur et de
# faire un random.sample par utilisateur → O(|U|) itérations Python,
# lent pour |U| > 10 000.
#
# Notre approche vectorisée :
#   1. Attribuer un nombre aléatoire à chaque rating
#   2. Trier par (user_id, aléatoire) pour mélanger intra-utilisateur
#   3. Calculer la position cumulative (rank) de chaque rating au sein
#      de son utilisateur via groupby().cumcount()
#   4. Calculer n_test pour chaque utilisateur de manière vectorisée
#   5. Les derniers n_test ratings de chaque utilisateur vont dans le
#      test, le reste dans le train
#
# Complexité : O(|R| log |R|) pour le tri, sans aucune boucle Python.
# Sur ~500 000 ratings, cela prend < 500 ms.
# ══════════════════════════════════════════════════════════════════════════


TRAIN_RATIO = 0.80
TEST_RATIO = 1 - TRAIN_RATIO
SEED = 42

splits = {}

for path in SAMPLE_GLOB_FILTERED:
    if os.path.getsize(path) < 1024:
        continue

    print(f"\n{'═' * 70}")
    print(f"  Split Train/Test : {path}")
    print(f"{'═' * 70}")

    df = pd.read_parquet(path)

    if len(df) == 0:
        print("  ⚠ Fichier vide, passage au suivant.")
        continue

    # Dédoublonner (même logique que la cellule matrice)
    n_dup = df.duplicated(subset=["user_id", "parent_asin"]).sum()
    if n_dup > 0:
        df = df.drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
        print(f"  {n_dup:,} doublons supprimés (même logique que la matrice CSR)")

    n_total = len(df)
    n_users = df["user_id"].nunique()
    n_items = df["parent_asin"].nunique()

    print(f"  Ratings : {n_total:,}  |  Utilisateurs : {n_users:,}  |  Livres : {n_items:,}")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : Mélange aléatoire intra-utilisateur (vectorisé)
    # ──────────────────────────────────────────────────────────────────
    # On attribue un nombre aléatoire à chaque ligne, puis on trie
    # par (user_id, aléatoire). Cela revient à faire un shuffle
    # indépendant des ratings de chaque utilisateur, mais en une
    # seule opération de tri sur tout le DataFrame.
    # ──────────────────────────────────────────────────────────────────

    t0 = time.perf_counter()

    rng = np.random.RandomState(SEED)
    df["_rand"] = rng.random(len(df))
    df = df.sort_values(["user_id", "_rand"]).reset_index(drop=True)

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : Calcul vectorisé de la position et du seuil de split
    # ──────────────────────────────────────────────────────────────────
    # Pour chaque utilisateur :
    #   - cumcount() : position 0, 1, 2, ... au sein de ses ratings
    #   - transform("count") : nombre total de ratings de cet utilisateur
    #   - n_test = max(1, floor(total × 0.2)), borné à total − 1
    #
    # Un rating est dans le TEST si sa position ≥ (total − n_test),
    # c'est-à-dire s'il fait partie des derniers n_test ratings
    # (après mélange aléatoire).
    # ──────────────────────────────────────────────────────────────────

    df["_pos"] = df.groupby("user_id").cumcount()
    df["_total"] = df.groupby("user_id")["_pos"].transform("count")

    n_test_per_user = np.floor(df["_total"].values * TEST_RATIO).astype(int)
    n_test_per_user = np.maximum(n_test_per_user, 1)          # au moins 1 en test
    n_test_per_user = np.minimum(n_test_per_user, df["_total"].values - 1)  # au moins 1 en train

    df["_n_test"] = n_test_per_user
    df["is_test"] = df["_pos"] >= (df["_total"] - df["_n_test"])

    train_df = df[~df["is_test"]].drop(columns=["_rand", "_pos", "_total", "_n_test", "is_test"])
    test_df = df[df["is_test"]].drop(columns=["_rand", "_pos", "_total", "_n_test", "is_test"])

    t_split = time.perf_counter() - t0

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : Validation de la contrainte de stratification
    # ──────────────────────────────────────────────────────────────────
    # On vérifie que CHAQUE utilisateur a au moins 1 rating dans le
    # train ET dans le test. C'est la contrainte fondamentale du split.
    # ──────────────────────────────────────────────────────────────────

    users_in_train = set(train_df["user_id"].unique())
    users_in_test = set(test_df["user_id"].unique())
    users_only_train = users_in_train - users_in_test
    users_only_test = users_in_test - users_in_train
    users_in_both = users_in_train & users_in_test

    print(f"\n  ── Étape 1-2 : Split vectorisé (seed={SEED}) ──")
    print(f"     Temps de calcul : {t_split * 1000:.1f} ms")
    print(f"     Ratio demandé   : {TRAIN_RATIO:.0%} train / {TEST_RATIO:.0%} test")

    actual_train_ratio = len(train_df) / n_total
    actual_test_ratio = len(test_df) / n_total

    print(f"     Ratio effectif  : {actual_train_ratio:.2%} train / {actual_test_ratio:.2%} test")
    if abs(actual_train_ratio - TRAIN_RATIO) > 0.02:
        print(f"     → Écart de {abs(actual_train_ratio - TRAIN_RATIO):.1%} par rapport au ratio cible.")
        print(f"       Cela est dû aux utilisateurs avec peu de ratings (n=2 ou 3)")
        print(f"       pour lesquels on est forcé de donner 1 rating au test,")
        print(f"       ce qui « sur-représente » légèrement le test.")
    else:
        print(f"     → Ratio respecté à ±2% près.")

    print(f"\n  ── Étape 3 : Validation de la stratification ──")
    print(f"     Utilisateurs dans train ET test  : {len(users_in_both):,}")
    print(f"     Utilisateurs SEULEMENT dans train: {len(users_only_train):,}")
    print(f"     Utilisateurs SEULEMENT dans test : {len(users_only_test):,}")

    if len(users_only_train) == 0 and len(users_only_test) == 0:
        print(f"     ✓ Contrainte respectée : chaque utilisateur a au moins 1 rating")
        print(f"       dans chaque ensemble.")
    else:
        print(f"     ⚠ VIOLATION : {len(users_only_train) + len(users_only_test)} "
              f"utilisateur(s) absent(s) d'un ensemble !")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 4 : Construction des matrices CSR train et test
    # ──────────────────────────────────────────────────────────────────
    # On réutilise pd.factorize sur l'ensemble COMPLET (train + test)
    # pour garantir que les indices utilisateur/livre sont cohérents
    # entre les deux matrices. Sinon, l'utilisateur 42 dans R_train
    # pourrait correspondre à un utilisateur différent dans R_test.
    # ──────────────────────────────────────────────────────────────────

    t1 = time.perf_counter()

    all_users = pd.concat([train_df["user_id"], test_df["user_id"]])
    all_items = pd.concat([train_df["parent_asin"], test_df["parent_asin"]])
    user_codes_all, user_ids = pd.factorize(all_users, sort=False)
    item_codes_all, item_ids = pd.factorize(all_items, sort=False)

    n_u = len(user_ids)
    n_i = len(item_ids)
    n_train = len(train_df)
    n_test = len(test_df)

    # Les n_train premiers codes correspondent au train, le reste au test
    train_user_codes = user_codes_all[:n_train]
    train_item_codes = item_codes_all[:n_train]
    test_user_codes = user_codes_all[n_train:]
    test_item_codes = item_codes_all[n_train:]

    R_train = csr_matrix(
        (train_df["rating"].values.astype(np.float32),
         (train_user_codes, train_item_codes)),
        shape=(n_u, n_i),
        dtype=np.float32,
    )

    R_test = csr_matrix(
        (test_df["rating"].values.astype(np.float32),
         (test_user_codes, test_item_codes)),
        shape=(n_u, n_i),
        dtype=np.float32,
    )

    t_build = time.perf_counter() - t1

    print(f"\n  ── Étape 4 : Matrices CSR train/test ──")
    print(f"     Dimensions communes : {n_u:,} × {n_i:,}")
    print(f"     R_train : {R_train.nnz:,} entrées  ({R_train.nnz / (n_u * n_i) * 100:.4f}% dense)")
    print(f"     R_test  : {R_test.nnz:,} entrées  ({R_test.nnz / (n_u * n_i) * 100:.4f}% dense)")
    print(f"     R_train + R_test = {R_train.nnz + R_test.nnz:,} "
          f"(total original : {n_total:,}, "
          f"après dédup : {n_train + n_test:,})")
    print(f"     Temps de construction : {t_build * 1000:.1f} ms")

    mem_train = R_train.data.nbytes + R_train.indices.nbytes + R_train.indptr.nbytes
    mem_test = R_test.data.nbytes + R_test.indices.nbytes + R_test.indptr.nbytes

    print(f"     Mémoire R_train : {mem_train / 1024**2:.2f} Mo")
    print(f"     Mémoire R_test  : {mem_test / 1024**2:.2f} Mo")

    # ──────────────────────────────────────────────────────────────────
    # ÉTAPE 5 : Distribution du nombre de ratings test par utilisateur
    # ──────────────────────────────────────────────────────────────────
    # On affiche la distribution pour vérifier que le split est
    # raisonnablement équilibré et que les cas extrêmes (n=2) sont
    # bien gérés.
    # ──────────────────────────────────────────────────────────────────

    test_per_user = test_df.groupby("user_id").size()
    train_per_user = train_df.groupby("user_id").size()

    print(f"\n  ── Étape 5 : Distribution du split par utilisateur ──")
    print(f"     Ratings TRAIN par utilisateur :")
    print(f"       min = {train_per_user.min()}  |  "
          f"médiane = {train_per_user.median():.0f}  |  "
          f"moyenne = {train_per_user.mean():.1f}  |  "
          f"max = {train_per_user.max()}")
    print(f"     Ratings TEST par utilisateur :")
    print(f"       min = {test_per_user.min()}  |  "
          f"médiane = {test_per_user.median():.0f}  |  "
          f"moyenne = {test_per_user.mean():.1f}  |  "
          f"max = {test_per_user.max()}")

    n_users_1_test = (test_per_user == 1).sum()
    print(f"     Utilisateurs avec exactement 1 rating test : {n_users_1_test:,}"
          f" ({n_users_1_test / n_users * 100:.1f}%)")
    print(f"     → Ce sont les utilisateurs avec ≤ 5 ratings originaux,")
    print(f"       pour lesquels floor(n × 0.2) ≤ 1.")

    # ──────────────────────────────────────────────────────────────────
    # Résumé final
    # ──────────────────────────────────────────────────────────────────

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DU SPLIT — {path}")
    print(f"  {'─' * 66}")
    print(f"  {'':30s} {'TRAIN':>14s}   {'TEST':>14s}   {'TOTAL':>10s}")
    print(f"  {'─' * 66}")
    print(f"  {'Ratings':30s} {n_train:>14,}   {n_test:>14,}   {n_train + n_test:>10,}")
    print(f"  {'Proportion':30s} {actual_train_ratio:>13.2%}   {actual_test_ratio:>13.2%}   {'100.00%':>10s}")
    print(f"  {'Utilisateurs présents':30s} {train_df['user_id'].nunique():>14,}   "
          f"{test_df['user_id'].nunique():>14,}   {n_users:>10,}")
    print(f"  {'Livres présents':30s} {train_df['parent_asin'].nunique():>14,}   "
          f"{test_df['parent_asin'].nunique():>14,}   {n_items:>10,}")
    print(f"  {'Mémoire CSR':30s} {mem_train / 1024**2:>13.2f}Mo   "
          f"{mem_test / 1024**2:>13.2f}Mo")
    print(f"  {'─' * 66}")

    # Stockage pour cellules suivantes
    splits[path] = {
        "R_train": R_train,
        "R_test": R_test,
        "train_df": train_df,
        "test_df": test_df,
        "user_ids": user_ids,
        "item_ids": item_ids,
    }

print(f"\n✓ {len(splits)} split(s) train/test construit(s) et stocké(s) dans `splits`.")

flush_ram()
flush_gpu()


#### B) Stockage des ensembles d'entraînement et de test

In [ ]:
flush_ram()
flush_gpu()

# ══════════════════════════════════════════════════════════════════════════
# SAUVEGARDE PERSISTANTE DES SPLITS TRAIN / TEST
#
# On écrit sur disque tous les artefacts nécessaires pour reprendre
# le travail sans avoir à ré-exécuter les cellules précédentes :
#
#   1. train.parquet / test.parquet
#      → Les DataFrames bruts (toutes les colonnes : user_id, parent_asin,
#        rating, timestamp, text, etc.). Format Parquet pour la portabilité
#        et la vitesse de lecture.
#
#   2. R_train.npz / R_test.npz
#      → Les matrices CSR au format natif scipy. Rechargement instantané
#        avec scipy.sparse.load_npz(), sans recalcul de pd.factorize
#        ni reconstruction de la matrice.
#
#   3. user_ids.npy / item_ids.npy
#      → Les tables de correspondance indice ↔ identifiant original.
#        Indispensables pour interpréter les résultats des modèles :
#          user_ids[42] → "AHJKFDS83KD" (ID Amazon)
#          item_ids[7]  → "B00005N5PF"  (ASIN du livre)
#
#   4. metadata.json
#      → Paramètres du split et statistiques clés. Permet de vérifier
#        la provenance et la cohérence des données sans les recharger.
#
# ── Structure sur disque ──────────────────────────────────────────────────
#
# Pour chaque échantillon (ex: sample-cudf-claude/), on crée un
# sous-dossier « splits/ » :
#
#   sample-cudf-claude/
#   ├── sample_gpu_active_users.parquet   ← fichier source (déjà existant)
#   └── splits/
#       ├── train.parquet                 ← 80% des ratings
#       ├── test.parquet                  ← 20% des ratings
#       ├── R_train.npz                   ← matrice CSR d'entraînement
#       ├── R_test.npz                    ← matrice CSR de test
#       ├── user_ids.npy                  ← mapping indice → user_id
#       ├── item_ids.npy                  ← mapping indice → parent_asin
#       └── metadata.json                ← paramètres et statistiques
#
# ── Pourquoi ce choix de formats ? ────────────────────────────────────────
#
# Parquet (DataFrames) :
#   - Compression columnar → 3-5× plus compact que CSV
#   - Préserve les types (float, int, string) sans ambiguïté
#   - Lisible par Pandas, Polars, DuckDB, Spark, etc.
#
# NPZ (matrices CSR) :
#   - Format natif de scipy.sparse → load_npz() reconstitue la CSR
#     en une seule instruction, sans re-factoriser les identifiants
#   - Stocke data[], indices[], indptr[] + shape en un seul fichier
#
# NPY (mappings) :
#   - Format natif NumPy, le plus rapide pour charger un array 1D
#   - allow_pickle=True nécessaire pour les arrays de strings
#
# JSON (métadonnées) :
#   - Lisible par un humain, facilement parsable
#   - Documente la provenance exacte des splits
# ══════════════════════════════════════════════════════════════════════════

for path, data in splits.items():
    sample_dir = Path(path).parent
    split_dir = sample_dir / "splits"
    split_dir.mkdir(exist_ok=True)

    print(f"\n{'═' * 70}")
    print(f"  Sauvegarde : {split_dir}/")
    print(f"{'═' * 70}")

    t0 = time.perf_counter()

    R_train = data["R_train"]
    R_test = data["R_test"]
    train_df = data["train_df"]
    test_df = data["test_df"]
    user_ids = data["user_ids"]
    item_ids = data["item_ids"]

    # ── 1. DataFrames Parquet ─────────────────────────────────────────

    train_path = split_dir / "train.parquet"
    test_path = split_dir / "test.parquet"

    train_df.to_parquet(train_path, index=False)
    test_df.to_parquet(test_path, index=False)

    size_train_pq = os.path.getsize(train_path)
    size_test_pq = os.path.getsize(test_path)

    print(f"\n  ── DataFrames Parquet ──")
    print(f"     {train_path.name:20s} : {len(train_df):>10,} lignes  |  {size_train_pq / 1024**2:>7.2f} Mo")
    print(f"     {test_path.name:20s} : {len(test_df):>10,} lignes  |  {size_test_pq / 1024**2:>7.2f} Mo")

    # ── 2. Matrices CSR (scipy NPZ) ──────────────────────────────────

    r_train_path = split_dir / "R_train.npz"
    r_test_path = split_dir / "R_test.npz"

    save_npz(r_train_path, R_train)
    save_npz(r_test_path, R_test)

    size_train_npz = os.path.getsize(r_train_path)
    size_test_npz = os.path.getsize(r_test_path)

    print(f"\n  ── Matrices CSR (NPZ) ──")
    print(f"     {r_train_path.name:20s} : {R_train.shape[0]:,}×{R_train.shape[1]:,}  "
          f"|  nnz={R_train.nnz:>10,}  |  {size_train_npz / 1024**2:>7.2f} Mo")
    print(f"     {r_test_path.name:20s} : {R_test.shape[0]:,}×{R_test.shape[1]:,}  "
          f"|  nnz={R_test.nnz:>10,}  |  {size_test_npz / 1024**2:>7.2f} Mo")

    # ── 3. Mappings (NumPy NPY) ──────────────────────────────────────

    user_path = split_dir / "user_ids.npy"
    item_path = split_dir / "item_ids.npy"

    np.save(user_path, user_ids)
    np.save(item_path, item_ids)

    size_user = os.path.getsize(user_path)
    size_item = os.path.getsize(item_path)

    print(f"\n  ── Mappings (NPY) ──")
    print(f"     {user_path.name:20s} : {len(user_ids):>10,} entrées  |  {size_user / 1024**2:>7.2f} Mo")
    print(f"     {item_path.name:20s} : {len(item_ids):>10,} entrées  |  {size_item / 1024**2:>7.2f} Mo")

    # ── 4. Métadonnées JSON ──────────────────────────────────────────

    metadata = {
        "source_file": str(path),
        "split_seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "test_ratio": TEST_RATIO,
        "n_users": int(len(user_ids)),
        "n_items": int(len(item_ids)),
        "train": {
            "n_ratings": int(R_train.nnz),
            "n_users": int(train_df["user_id"].nunique()),
            "n_items": int(train_df["parent_asin"].nunique()),
            "sparsity": float(1 - R_train.nnz / (len(user_ids) * len(item_ids))),
            "file_parquet": "train.parquet",
            "file_csr": "R_train.npz",
        },
        "test": {
            "n_ratings": int(R_test.nnz),
            "n_users": int(test_df["user_id"].nunique()),
            "n_items": int(test_df["parent_asin"].nunique()),
            "sparsity": float(1 - R_test.nnz / (len(user_ids) * len(item_ids))),
            "file_parquet": "test.parquet",
            "file_csr": "R_test.npz",
        },
        "mappings": {
            "file_user_ids": "user_ids.npy",
            "file_item_ids": "item_ids.npy",
        },
    }

    meta_path = split_dir / "metadata.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print(f"\n  ── Métadonnées ──")
    print(f"     {meta_path.name:20s} : paramètres du split + statistiques")

    # ── Résumé ────────────────────────────────────────────────────────

    t_save = time.perf_counter() - t0
    total_size = size_train_pq + size_test_pq + size_train_npz + size_test_npz + size_user + size_item

    print(f"\n  {'─' * 66}")
    print(f"  RÉSUMÉ DE LA SAUVEGARDE — {split_dir}/")
    print(f"  {'─' * 66}")
    print(f"     Fichiers écrits  : 7")
    print(f"     Taille totale    : {total_size / 1024**2:.2f} Mo")
    print(f"     Temps d'écriture : {t_save * 1000:.1f} ms")
    print(f"  {'─' * 66}")
    print(f"     Contenu du dossier :")
    for f in sorted(split_dir.iterdir()):
        print(f"       {f.name:25s}  {os.path.getsize(f) / 1024**2:>7.2f} Mo")
    print(f"  {'─' * 66}")

print(f"\n✓ Tous les splits sont sauvegardés sur disque.")
print(f"\n  Pour recharger dans un autre notebook :")
print(f"  ┌─────────────────────────────────────────────────────────────────┐")
print(f"  │  from scipy.sparse import load_npz                            │")
print(f"  │  import numpy as np, pandas as pd, json                       │")
print(f"  │                                                               │")
print(f"  │  SPLIT = ''                          │")
print(f"  │                                                               │")
print(f"  │  train_df = pd.read_parquet(f'{{SPLIT}}/train.parquet')        │")
print(f"  │  test_df  = pd.read_parquet(f'{{SPLIT}}/test.parquet')         │")
print(f"  │  R_train  = load_npz(f'{{SPLIT}}/R_train.npz')                │")
print(f"  │  R_test   = load_npz(f'{{SPLIT}}/R_test.npz')                 │")
print(f"  │  user_ids = np.load(f'{{SPLIT}}/user_ids.npy',                │")
print(f"  │                     allow_pickle=True)                        │")
print(f"  │  item_ids = np.load(f'{{SPLIT}}/item_ids.npy',                │")
print(f"  │                     allow_pickle=True)                        │")
print(f"  │  meta     = json.load(open(f'{{SPLIT}}/metadata.json'))        │")
print(f"  └─────────────────────────────────────────────────────────────────┘")